# News Forecasting — Model Consensus Analysis

Compare how frontier LLMs forecast real-world outcomes from news articles, with a focus on **model consensus**: which questions do all models agree on, and where do they diverge?

The full pipeline:
1. **Collect news** articles from a date range
2. **Generate questions** — binary yes/no forecasting questions about future events
3. **Find answers** — web search discovers what actually happened
4. **Render prompts** — format each question for model consumption
5. **Send to models** — multiple LLMs produce probability forecasts
6. **Score** — compare each model's forecasts against ground truth

In [2]:
%pip install lightningrod-ai python-dotenv pandas

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [3]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure the pipeline

The pipeline has six stages:

1. **NewsSeedGenerator** — collects news articles from a date range
2. **ForwardLookingQuestionGenerator** — creates binary yes/no forecasting questions from those articles
3. **WebSearchLabeler** — finds ground-truth answers by searching for what actually happened
4. **QuestionRenderer** — renders the final prompt with answer format instructions
5. **RolloutGenerator** — sends the rendered prompt to multiple LLMs for comparison
6. **RolloutScorer** — scores each model's probability forecast against the ground-truth label

We use dates a few months in the past so the questions can be resolved.

In [ ]:
from datetime import datetime
from lightningrod import (
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    WebSearchLabeler,
    QuestionPipeline,
    NewsContextGenerator,
    QuestionRenderer,
    RolloutGenerator,
    RolloutScorer,
    BinaryAnswerType,
    open_router_model,
)

# Date range — adjust these to a period ~2-3 months in the past
START_DATE = datetime(2025, 11, 1)
END_DATE = datetime(2025, 12, 1)

seed_generator = NewsSeedGenerator(
    start_date=START_DATE,
    end_date=END_DATE,
    search_query="technology announcements",
)

answer_type = BinaryAnswerType()

question_generator = ForwardLookingQuestionGenerator(
    instructions="Generate forward-looking yes/no questions about technology announcements. "
    "Questions should be clearly resolvable within 1-2 months.",
    answer_type=answer_type,
)

labeler = WebSearchLabeler(answer_type=answer_type)

renderer = QuestionRenderer(answer_type=answer_type)

models = [
    open_router_model("openai/gpt-4.1-mini"),
    open_router_model("anthropic/claude-sonnet-4"),
    open_router_model("google/gemini-2.5-flash"),
]

context_generator = NewsContextGenerator()

rollout_generator = RolloutGenerator(models=models)

scorer = RolloutScorer(answer_type=answer_type)

pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    context_generators=[context_generator],
    labeler=labeler,
    renderer=renderer,
    rollout_generator=rollout_generator,
    scorer=scorer,
)

## Run the pipeline

> Note: This can take several minutes — the pipeline collects news, generates questions, searches for answers, then sends each question to all three models.

In [5]:
dataset = lr.transforms.run(pipeline, max_seeds=20, name="News Forecasting Benchmark")  # Increase to ~10000 for a real run

Output()

## View generated questions and labels

Each sample contains a forecasting question generated from a news article and a ground-truth label found via web search.

In [6]:
samples = dataset.download()

print(f"Generated {dataset.num_rows} samples ({dataset.valid_count() / dataset.num_rows * 100:.1f}% valid)\n")


Generated 20 samples (80.0% valid)



## Consensus analysis

Where do the models agree, and where do they diverge? `compute_consensus` extracts each model's predicted probability and computes:
- **spread** — max probability minus min probability across models (higher = more disagreement)
- **all_agree** — whether all models predict the same side of 0.5

In [7]:
from lightningrod.utils import compute_consensus
import pandas as pd

consensus = compute_consensus(samples)
n_agree = sum(1 for c in consensus if c["all_agree"])
n_total = len(consensus)

print(f"Consensus: {n_agree}/{n_total} questions have full agreement ({n_agree / n_total * 100:.0f}%)")
print(f"Disagreement: {n_total - n_agree}/{n_total} questions have models on opposite sides of 0.5")
print(f"Mean spread: {sum(c['spread'] for c in consensus) / n_total:.3f}")
print()

# Build a DataFrame with per-model predictions and spread
rows = []
for c in consensus:
    row = {"Question": c["question_text"], "Label": c["label"], "Spread": round(c["spread"], 3), "Agree": c["all_agree"]}
    for model, prob in c["predictions"].items():
        short_name = model.split("/")[-1] if "/" in model else model
        row[short_name] = round(prob, 3)
    rows.append(row)

df_consensus = pd.DataFrame(rows)
df_consensus

Consensus: 6/16 questions have full agreement (38%)
Disagreement: 10/16 questions have models on opposite sides of 0.5
Mean spread: 0.400



,Question,Label,Spread,Agree,gpt-4.1-mini,claude-sonnet-4,gemini-2.5-flash
0,Will the New Frontiers in Research Fund (NFRF)...,1,0.719,False,0.001,0.72,0.01
1,Will Autozi Internet Technology (AZI) complete...,1,0.670,False,0.050,0.72,0.50
2,Is the 13th Seoul Mediacity Biennale catalogue...,1,0.670,False,0.050,0.72,0.50
3,Will Mercury Ev-Tech Limited hold its 39th Ann...,0,0.620,False,0.600,0.72,0.10
4,Will Supermicro announce or list at least one ...,1,0.500,False,0.150,0.65,0.60
5,Will the Commonwealth Bank of Australia (CBA) ...,0,0.450,False,0.200,0.65,0.60
6,"By January 31, 2026, will Trimble have officia...",1,0.420,False,0.450,0.72,0.30
7,Will the application period for Innovate UK's ...,0,0.400,True,0.900,0.65,0.50
8,Will the weekly U.S. initial jobless claims (s...,0,0.400,False,0.200,0.35,0.60
9,"By December 31, 2025, will Google officially r...",1,0.400,False,0.200,0.35,0.60


## Per-model accuracy (secondary)

For reference, here are the individual model metrics. `mean_reward` captures how well-calibrated each model's probability estimates are (higher is better).

In [8]:
from lightningrod.utils import compute_metrics_summary

summary = compute_metrics_summary(samples)
df_metrics = pd.DataFrame.from_dict(summary, orient="index")
df_metrics.index.name = "model"
df_metrics[["mean_reward", "parse_rate", "n_total"]]

,mean_reward,parse_rate,n_total
model,,,
openai/gpt-4.1-mini,-0.443781,1.0,16
anthropic/claude-sonnet-4,-0.270962,1.0,16
google/gemini-2.5-flash,-0.331725,1.0,16


## Next steps

- **Quick start**: See [01_quick_start.ipynb](../01_quick_start.ipynb) for the basic news forecasting pipeline
- **Document classification benchmark**: See [document_classification_benchmark.ipynb](document_classification_benchmark.ipynb) for benchmarking on a pre-existing dataset
- **Different question types**: See examples 04-07 for different question and answer types
- **Full API reference**: See [API.md](../../API.md) for all options and configurations